# Haiku Multiple Instance Learning

**Project Name:** Haiku

## Purpose
- Publication-ready notebook for reproducible evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

# Notebook lives at <repo>/downstream/ — HAIKU_ROOT points at the repo root.
HAIKU_ROOT = Path.cwd().parent
if str(HAIKU_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(HAIKU_ROOT / 'src'))

# -------- Fill these in to point at your local copies --------
EMBEDDINGS_DIR = Path('<PATH_TO_PRECOMPUTED_EMBEDDINGS>')
SAMPLES_JSON   = HAIKU_ROOT / 'overlap_samples_final.json'
OUTPUT_DIR     = HAIKU_ROOT / 'downstream' / 'figs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root=str(HAIKU_ROOT))
seed_everything(42)


In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch.nn.functional as F
import json
import sys


codex_embedding = torch.load(EMBEDDINGS_DIR / 'new_codex_embedding.pt')

virtual_codex_embedding = torch.load(EMBEDDINGS_DIR / 'new_virtual_codex_embedding.pt')

region_label = torch.load(EMBEDDINGS_DIR / 'new_region_label.pt')

print(codex_embedding.shape)

sample_ids = list(json.load(open(SAMPLES_JSON)).keys())

ref_ids = sorted(sample_ids)

# Build bags of embeddings for each patient
# The mapping from region_label[i] -> ref_ids[region_label[i]] will give you a region/acquisition ID
patient_he_bags = dict()        # patient_id: [he_embedding_tensors]
patient_codex_bags = dict()  # patient_id: [codex_embedding_tensors]
patient_virt_bags = dict()   # patient_id: [virtual_codex_embedding_tensors]
patient_musk_bags = dict()   # patient_id: [musk_embedding_tensors]
patient_labels = dict()
patient_concat_bags = dict()    # patient_id: label (list or scalar per patient, depending)

for idx, region_l in enumerate(region_label):
    # Lookup the patient_id for this region/acquisition

    region_id = ref_ids[region_l]

    # Group embeddings by patient
    #patient_he_bags.setdefault(region_id, []).append(he_embedding[idx])
    patient_codex_bags.setdefault(region_id, []).append(codex_embedding[idx])
    patient_virt_bags.setdefault(region_id, []).append(virtual_codex_embedding[idx])
    #patient_musk_bags.setdefault(region_id, []).append(musk_embedding[idx])
    #patient_concat_bags.setdefault(region_id, []).append(torch.cat([he_embedding[idx], codex_embedding[idx]], dim=0))

    # Optionally: also maintain region-labels or per-patient labels.
    # Here, defaulting to using the region's integer label.
    patient_labels.setdefault(region_id, []).append(region_label[idx])

# Optionally, if you want a "single label per patient" for classification (e.g. majority, first, or a function):
# Example (majority label per patient):
from collections import Counter
patient_majority_labels = {
    pid: Counter(lbls).most_common(1)[0][0]
    for pid, lbls in patient_labels.items()
}


In [ ]:
import os
import pandas as pd

csv_list = [
    HAIKU_ROOT / 'downstream' / 'Lymphoma_response-to-RCHOP.csv',
    HAIKU_ROOT / 'downstream' / 'Melanoma_response-to-immunotherapy.csv',
    HAIKU_ROOT / 'downstream' / 'CRC_terminal_survival.csv',
]


need_new_acq_ids = []

df_list = []

for i in csv_list:
    df = pd.read_csv(i)
    new_acq_ids = df['ACQUISITION_ID'].tolist()
    df_list.append(df)


In [ ]:
survial_length_dict = {}
survial_status_dict = { }
response_dict = {}
treatment_dict = {}

In [ ]:
survial_length_dict['1'] = 'survival'
survial_status_dict['1'] = 'survival_status'
response_dict['1'] = 'Response-binary'
treatment_dict['1'] = 'treatment'

survial_length_dict['2'] = 'FOLLOW UP (months)'
survial_status_dict['2'] = 'survival_status'
response_dict['2'] = 'Outcome-binary'
treatment_dict['2'] = 'treatment'

In [ ]:
# End-to-end Multiple-Instance Learning (MIL) with Cox loss:
# - Splits into train, val, *test* and reports test C-index & KM
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple
import random
import matplotlib.pyplot as plt
import pandas as pd

from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test

# ----------------------------
# 0) Reproducibility
# ----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---------------------------------------
# 1) Expected input dicts (replace these)
# ---------------------------------------
# These three dicts + four embedding dicts (for four methods/models now):
# time_dict  : {bag_id: float_survival_time}
# event_dict : {bag_id: int_event (1=death, 0=censored)}
# baseline_embeddings : {bag_id: [np.ndarray (Din,), ...]}
# our_embeddings      : {bag_id: [np.ndarray (Din,), ...]}
# third_method_embeddings : {bag_id: [np.ndarray (Din,), ...]}
# fourth_method_embeddings: {bag_id: [np.ndarray (Din,), ...]}


index = 2

df = df_list[index]

region_labels_survival = {}
region_values_survival_status = {}

for region_id in sample_ids:
    if region_id in df['ACQUISITION_ID'].values:
        region_labels_survival[region_id] = df[df['ACQUISITION_ID'] == region_id][survial_length_dict[str(index)]].values
        region_values_survival_status[region_id] = df[df['ACQUISITION_ID'] == region_id][survial_status_dict[str(index)]].values

interaction_region_ids = set(region_labels_survival.keys()) & set(region_values_survival_status.keys())

# If you want DataFrames/labels only for these interactions:
region_labels_survival_interaction = {rid: float(region_labels_survival[rid]) for rid in interaction_region_ids}
region_values_survival_status_interaction = {rid: int(region_values_survival_status[rid] != 'Alive') for rid in interaction_region_ids}

time_dict = region_labels_survival_interaction
event_dict = region_values_survival_status_interaction
baseline_embeddings = patient_virt_bags
our_embeddings = patient_codex_bags

print(len(list(interaction_region_ids)))

In [ ]:
# ============================================================
# End-to-end MIL-Cox: 5-fold CV (Baseline vs Ours)
# - Trains per fold with cosine LR + patience early-stop + best-state restore
# - Reports per-fold Test C-index
# - Plots C-index boxplot (TEST)
# - Saves per-fold KM curves + log-rank p-values (TEST)
# Mirrors examples/mil_surv.py — never leaks test into val.
# ============================================================
import os, json, random, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from typing import Dict, List
import matplotlib.pyplot as plt
import pandas as pd

from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test

from sklearn.model_selection import StratifiedKFold, train_test_split

# --------------------- Matplotlib: SVG text editable ---------------------
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42

# --------------------- Reproducibility ---------------------
SEED = 42
N_FOLDS = 5

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------- Best HPs (mil_surv.py task 2 / CRC) ---------------------
BEST_HP_SURV = {
    "embed_dim": 128, "pool": "attn", "lr": 1e-3, "dropout": 0.0,
    "l2_norm": True, "batch_size": 64, "wd": 0.0, "max_epochs": 50, "patience": 10,
}

# =====================================================================
# REQUIRED INPUTS (provided by upstream cells):
#   time_dict, event_dict, baseline_embeddings, our_embeddings
# =====================================================================

# =====================================================================
# Dataset & collate
# =====================================================================
def l2_normalize(x, axis=1, eps=1e-12):
    norm = np.linalg.norm(x, ord=2, axis=axis, keepdims=True)
    return x / np.clip(norm, a_min=eps, a_max=None)


class MILDataset(Dataset):
    def __init__(self, emb_dict: Dict[str, List[np.ndarray]],
                 time_dict: Dict[str, float],
                 event_dict: Dict[str, int],
                 bag_ids: List[str],
                 do_l2_norm: bool = True):
        self.ids, self.bags, self.times, self.events = [], [], [], []
        for bid in bag_ids:
            if bid not in emb_dict or bid not in time_dict or bid not in event_dict:
                continue
            raw = emb_dict[bid]
            if isinstance(raw, np.ndarray) and raw.ndim == 2:
                X = raw.astype(np.float32)
            elif isinstance(raw, torch.Tensor) and raw.ndim == 2:
                X = raw.numpy().astype(np.float32)
            else:
                if len(raw) == 0:
                    continue
                X = np.stack([r.numpy() if torch.is_tensor(r) else np.asarray(r)
                              for r in raw], axis=0).astype(np.float32)
            if do_l2_norm:
                X = l2_normalize(X, axis=1)
            self.ids.append(bid)
            self.bags.append(torch.from_numpy(X))
            self.times.append(float(time_dict[bid]))
            self.events.append(int(event_dict[bid]))
        if len(self.bags) == 0:
            raise ValueError("Empty MILDataset \u2014 check input dicts/split.")
        self.Din = self.bags[0].shape[1]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return (
            self.bags[i],
            torch.tensor(self.times[i], dtype=torch.float32),
            torch.tensor(self.events[i], dtype=torch.long),
            self.ids[i],
        )


def pad_collate(batch):
    bags, times, events, ids = zip(*batch)
    B = len(bags); Nmax = max(b.shape[0] for b in bags); D = bags[0].shape[1]
    X = torch.zeros(B, Nmax, D, dtype=torch.float32)
    M = torch.zeros(B, Nmax, dtype=torch.bool)
    for i, b in enumerate(bags):
        n = b.shape[0]
        X[i, :n] = b
        M[i, :n] = True
    t = torch.stack(times)
    e = torch.stack(events)
    return X, M, t, e, list(ids)

# =====================================================================
# MIL-Cox model & loss
# =====================================================================
def cox_partial_ll(risk: torch.Tensor, time: torch.Tensor, event: torch.Tensor):
    order = torch.argsort(time, descending=True)
    risk  = risk[order]
    event = event[order].float()
    log_cumsum_exp = torch.logcumsumexp(risk, dim=0)
    ll = event * (risk - log_cumsum_exp)
    denom = event.sum().clamp_min(1.0)
    return -ll.sum() / denom


class GatedAttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.U = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H)) * torch.sigmoid(self.U(H))).squeeze(-1)
        A = A.masked_fill(~mask, float("-inf"))
        A = torch.softmax(A, dim=1)
        return torch.einsum("bn,bnd->bd", A, H), A


class AttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H))).squeeze(-1)
        A = A.masked_fill(~mask, float("-inf"))
        A = torch.softmax(A, dim=1)
        return torch.einsum("bn,bnd->bd", A, H), A


class MILCox(nn.Module):
    def __init__(self, in_dim, embed_dim=128, pool="attn", dropout=0.0):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, embed_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim), nn.GELU(),
            nn.LayerNorm(embed_dim),
        )
        self.pool_type = pool
        if pool == "gated_attn":
            self.pool = GatedAttnPool(embed_dim)
        elif pool == "attn":
            self.pool = AttnPool(embed_dim)
        else:
            self.pool = None
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, X, mask):
        H = self.encoder(X)
        if self.pool_type in ("attn", "gated_attn"):
            Z, A = self.pool(H, mask)
        elif self.pool_type == "mean":
            Z = (H * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1).to(H.dtype)
            A = None
        else:
            raise ValueError(f"Unknown pool: {self.pool_type}")
        risk = self.head(Z).squeeze(-1)  # higher = worse
        return risk, A

# =====================================================================
# Train / predict (with cosine LR + patience + best-state restore)
# =====================================================================
def train_end2end_milcox(emb_dict, time_dict, event_dict, train_ids, val_ids, hp,
                         device=device, seed=SEED):
    """Mirrors examples/mil_surv.py:train_model."""
    set_seed(seed)
    train_ds = MILDataset(emb_dict, time_dict, event_dict, train_ids, do_l2_norm=hp["l2_norm"])
    val_ds   = MILDataset(emb_dict, time_dict, event_dict, val_ids,   do_l2_norm=hp["l2_norm"])

    train_loader = DataLoader(train_ds, batch_size=hp["batch_size"], shuffle=True,
                              collate_fn=pad_collate, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=hp["batch_size"], shuffle=False,
                              collate_fn=pad_collate, num_workers=0)

    model = MILCox(train_ds.Din, hp["embed_dim"], hp["pool"], hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
    scheduler = CosineAnnealingLR(opt, T_max=hp["max_epochs"], eta_min=hp["lr"] * 0.01)

    best_val_cidx = -1.0
    best_state = None
    patience_counter = 0

    for epoch in range(1, hp["max_epochs"] + 1):
        model.train()
        for X, M, T, E, _ in train_loader:
            X, M, T, E = X.to(device), M.to(device), T.to(device), E.to(device)
            risk, _ = model(X, M)
            loss = cox_partial_ll(risk, T, E)
            opt.zero_grad(); loss.backward(); opt.step()
        scheduler.step()

        model.eval()
        vt, ve, vr = [], [], []
        with torch.no_grad():
            for X, M, T, E, _ in val_loader:
                r, _ = model(X.to(device), M.to(device))
                vt.append(T.numpy())
                ve.append(E.numpy())
                vr.append(r.cpu().numpy())
        vt = np.concatenate(vt); ve = np.concatenate(ve); vr = np.concatenate(vr)

        try:
            cidx = concordance_index(vt, -vr, ve)
        except Exception:
            cidx = 0.5

        if cidx > best_val_cidx:
            best_val_cidx = cidx
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= hp["patience"]:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_df(model, emb_dict, time_dict, event_dict, ids, hp, device=device):
    ds = MILDataset(emb_dict, time_dict, event_dict, ids, do_l2_norm=hp["l2_norm"])
    loader = DataLoader(ds, batch_size=256, shuffle=False, collate_fn=pad_collate)
    out_ids, out_t, out_e, out_r = [], [], [], []
    model.eval()
    with torch.no_grad():
        for X, M, T, E, ID in loader:
            r, _ = model(X.to(device), M.to(device))
            out_ids += ID
            out_t   += T.tolist()
            out_e   += E.tolist()
            out_r   += r.cpu().numpy().tolist()
    return pd.DataFrame({"bag_id": out_ids, "time": out_t, "event": out_e, "risk": out_r})

# =====================================================================
# KM helpers (per-fold curves + log-rank p-value)
# =====================================================================
def km_curves_and_logrank(df: pd.DataFrame):
    risk = df["risk"].values
    time = df["time"].values
    event = df["event"].values

    med = np.median(risk)
    grp = np.where(risk >= med, "High", "Low")

    curves = {}
    km = KaplanMeierFitter()
    for label in ["Low", "High"]:
        m = (grp == label)
        if m.sum() < 2:
            curves[label] = (np.array([0.0, 1.0]), np.array([1.0, 1.0]))
            continue
        km.fit(durations=time[m], event_observed=event[m], label=label)
        t = km.survival_function_.index.values.astype(float)
        s = km.survival_function_[label].values.astype(float)
        if t[0] > 0:
            t = np.insert(t, 0, 0.0)
            s = np.insert(s, 0, 1.0)
        curves[label] = (t, s)

    low_mask = (grp == "Low")
    high_mask = (grp == "High")
    if low_mask.sum() >= 2 and high_mask.sum() >= 2:
        res = logrank_test(
            time[low_mask], time[high_mask],
            event_observed_A=event[low_mask],
            event_observed_B=event[high_mask],
        )
        p_value = float(res.p_value)
    else:
        p_value = float("nan")

    return curves, p_value

# =====================================================================
# 5-fold CV driver (Baseline vs Ours)
# =====================================================================
def run_cv5_milcox(time_dict, event_dict, baseline_embeddings, our_embeddings, hp,
                   device=device, seed=SEED):
    """Mirrors examples/mil_surv.py:run_cv5. Takes a single hp dict; never leaks test into val."""
    set_seed(seed)

    common_ids = set(time_dict) & set(event_dict) & set(baseline_embeddings) & set(our_embeddings)
    all_ids = sorted(common_ids)
    if len(all_ids) < 5:
        raise ValueError("Not enough bags for 5-fold CV.")
    y_event = np.array([event_dict[i] for i in all_ids])
    y_time  = np.array([time_dict[i]  for i in all_ids])

    q = np.quantile(y_time, [0.25, 0.5, 0.75])
    time_bucket = np.digitize(y_time, q)
    strata = y_event.astype(str) + "_" + time_bucket.astype(str)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    base_cidx_folds, ours_cidx_folds = [], []
    base_tests_folds, ours_tests_folds = [], []
    base_km_pvals, ours_km_pvals = [], []

    fold = 0
    for tr_idx, te_idx in skf.split(all_ids, strata):
        fold += 1
        train_ids = [all_ids[i] for i in tr_idx]
        test_ids  = [all_ids[i] for i in te_idx]

        y_tr_event = np.array([event_dict[i] for i in train_ids])
        # No leakage: skip fold rather than fall back to test as val
        if len(train_ids) < 20 or len(np.unique(y_tr_event)) < 2:
            continue

        tr_ids, va_ids = train_test_split(
            train_ids,
            test_size=max(2, int(0.1 * len(train_ids))),
            random_state=seed,
            stratify=y_tr_event,
        )

        # ---- Train Baseline ----
        base_model = train_end2end_milcox(
            baseline_embeddings, time_dict, event_dict, tr_ids, va_ids, hp,
            device=device, seed=seed,
        )
        base_test = predict_df(base_model, baseline_embeddings, time_dict, event_dict, test_ids,
                               hp, device=device)
        cidx_b = concordance_index(
            base_test["time"].values,
            -np.asarray(base_test["risk"].values),
            base_test["event"].values,
        )
        base_cidx_folds.append(float(cidx_b))
        base_tests_folds.append(base_test)

        base_curves, base_p = km_curves_and_logrank(base_test)
        base_km_pvals.append(base_p)

        # ---- Train Ours ----
        ours_model = train_end2end_milcox(
            our_embeddings, time_dict, event_dict, tr_ids, va_ids, hp,
            device=device, seed=seed,
        )
        ours_test = predict_df(ours_model, our_embeddings, time_dict, event_dict, test_ids,
                               hp, device=device)
        cidx_o = concordance_index(
            ours_test["time"].values,
            -np.asarray(ours_test["risk"].values),
            ours_test["event"].values,
        )
        ours_cidx_folds.append(float(cidx_o))
        ours_tests_folds.append(ours_test)

        ours_curves, ours_p = km_curves_and_logrank(ours_test)
        ours_km_pvals.append(ours_p)

    # ---------------- C-index boxplot ----------------
    plt.figure(figsize=(7.2, 4.6))
    plt.boxplot(
        [base_cidx_folds, ours_cidx_folds],
        labels=["Baseline", "Ours"],
        showmeans=False,
    )
    plt.ylabel("C-index (higher is better)")
    plt.title("MIL-Cox \u2014 5-fold Test C-index")
    plt.ylim(0.4, 1.0)
    for i, arr in enumerate([base_cidx_folds, ours_cidx_folds], start=1):
        plt.text(i, np.nanmean(arr) + 0.02, f"{np.nanmean(arr):.3f}", ha="center", fontsize=10)
    plt.tight_layout()
    plt.show()

    return {
        "fold_cindex": {"Baseline": base_cidx_folds, "Ours": ours_cidx_folds},
        "km_logrank_p": {"Baseline": base_km_pvals, "Ours": ours_km_pvals},
        "base_tests_folds": base_tests_folds,
        "ours_tests_folds": ours_tests_folds,
        "hyperparameters": hp,
        "meta": {"n_ids": len(all_ids), "methods": ["Baseline", "Ours"], "n_folds": N_FOLDS},
    }


artifacts = run_cv5_milcox(
    time_dict=time_dict,
    event_dict=event_dict,
    baseline_embeddings=baseline_embeddings,
    our_embeddings=our_embeddings,
    hp=BEST_HP_SURV,
    device=device,
    seed=SEED,
)
